# CBUS mini-reproduction: Kaggle session 2 (Qwen2.5-7B, free GPU)
Settings: Accelerator **GPU T4 x2**, **Internet ON**. Create this as a NEW notebook/session (the first session's server, data and cache are gone).
Run the cells top to bottom. **Stages are independent: after each stage there is an export cell; download `outputs.zip` (right panel > Output) before moving on**, so a crash or quota stop never loses finished work.
Approximate time at N=300 (from your pilot speeds, a guess): diagnostic 5 min, stage 2 about 50 min, stage 3 about 2 h, stage 4 about 30 min, stage 5 about 1.5 h.

## 0. Setup (same as session 1)

In [1]:
import os, subprocess, shutil, json, glob
REPO = 'https://github.com/sarrazer24/cbus-repro'
if not os.path.exists('/kaggle/working/cbus-repro'):
    subprocess.run(['git','clone',REPO,'/kaggle/working/cbus-repro'],check=True)
if not os.path.exists('/tmp/authors_data'):
    subprocess.run(['git','clone','--depth','1','https://github.com/iai-group/emnlp2026-simreading','/tmp/authors_data'],check=True)
os.chdir('/kaggle/working/cbus-repro')
os.environ['SIMREADING_DATA'] = '/tmp/authors_data'
subprocess.run('pip install -q -r requirements.txt',shell=True)
print(os.getcwd())

Cloning into '/kaggle/working/cbus-repro'...
Cloning into '/tmp/authors_data'...


/kaggle/working/cbus-repro


In [2]:
!nvidia-smi

Thu Nov 12 19:49:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install -q vllm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.3.0 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


## 1. Start the server (about 5 min). If it exits, run the log cell and send me the output.

In [4]:
import time, requests
MODEL = 'Qwen/Qwen2.5-7B-Instruct'; TP = 2; NAME = 'local'
log = open('/tmp/vllm.log','w')
env = dict(os.environ, VLLM_ATTENTION_BACKEND='XFORMERS')
proc = subprocess.Popen(['python','-m','vllm.entrypoints.openai.api_server','--model',MODEL,'--served-model-name',NAME,
    '--dtype','half','--tensor-parallel-size',str(TP),'--max-model-len','8192','--gpu-memory-utilization','0.90',
    '--enable-prefix-caching','--port','8000'], stdout=log, stderr=subprocess.STDOUT, env=env)
for i in range(120):
    try:
        if requests.get('http://localhost:8000/v1/models',timeout=2).ok: print('server ready after',i*5,'s'); break
    except Exception: pass
    if proc.poll() is not None: print('server exited; see /tmp/vllm.log'); break
    time.sleep(5)

server ready after 300 s


In [5]:
!tail -n 20 /tmp/vllm.log

(APIServer pid=103) INFO 11-12 19:58:49 [launcher.py:197] Route: /v1/responses/{response_id}, Methods: GET
(APIServer pid=103) INFO 11-12 19:58:49 [launcher.py:197] Route: /v1/responses/{response_id}/cancel, Methods: POST
(APIServer pid=103) INFO 11-12 19:58:49 [launcher.py:197] Route: /v1/completions, Methods: POST
(APIServer pid=103) INFO 11-12 19:58:49 [launcher.py:197] Route: /v1/messages, Methods: POST
(APIServer pid=103) INFO 11-12 19:58:49 [launcher.py:197] Route: /v1/messages/count_tokens, Methods: POST
(APIServer pid=103) INFO 11-12 19:58:49 [launcher.py:197] Route: /generative_scoring, Methods: POST
(APIServer pid=103) INFO 11-12 19:58:49 [launcher.py:197] Route: /scale_elastic_ep, Methods: POST
(APIServer pid=103) INFO 11-12 19:58:49 [launcher.py:197] Route: /is_scaling_elastic_ep, Methods: POST
(APIServer pid=103) INFO 11-12 19:58:49 [launcher.py:197] Route: /v1/chat/completions/render, Methods: POST
(APIServer pid=103) INFO 11-12 19:58:49 [launcher.py:197] Route: /v1/messa

## 2. Config and helpers
`MT` = max output tokens for all NEW runs (session 1 used 400; replies are short JSON, 1000 only protects long proposition lists from truncation). Report this difference in the post.

In [6]:
MT = 1000
COMMON = f'--base-url http://localhost:8000/v1 --model local --max-tokens {MT} --workers 32 --cache /tmp/cache.jsonl'
COMMON400 = COMMON.replace(f'--max-tokens {MT}','--max-tokens 400')
def export():
    shutil.make_archive('/kaggle/working/outputs','zip','.','runs')
    print('exported ->', '/kaggle/working/outputs.zip (download it now)')
def show(pattern):
    for f in sorted(glob.glob(pattern)):
        m=json.load(open(f)); print(m['name'],'| dropped_stage1',m['dropped_stage1_calls'],'| dropped_answers',m['dropped_answer_replies'],'| seconds',m['seconds'],'|',m['usage'])
print(COMMON)

--base-url http://localhost:8000/v1 --model local --max-tokens 1000 --workers 32 --cache /tmp/cache.jsonl


## Stage 1: truncation diagnostic (20 students, about 5 min)
Does asking for 12 propositions with a 400-token limit cut the JSON off? Compare the two printed lines and send them to me.

In [7]:
!python scripts/02_run_simulation.py --mode spr_dropout --students 20 {COMMON400} --out-root runs_pilot --name d400
!python scripts/02_run_simulation.py --mode spr_dropout --students 20 {COMMON} --out-root runs_pilot --name d1000
show('runs_pilot/d*/meta.json')

[run] d400: 20 students x ~7 texts
  14/140 student-texts
  28/140 student-texts
  42/140 student-texts
  56/140 student-texts
  70/140 student-texts
  84/140 student-texts
  98/140 student-texts
  112/140 student-texts
  126/140 student-texts
  140/140 student-texts
[done] answered=660 dropped=20 (2.94%) usage={'calls': 802, 'cache_hits': 0, 'input_tokens': 299144, 'output_tokens': 32759} -> runs_pilot/d400
[run] d1000: 20 students x ~7 texts
  14/140 student-texts
  28/140 student-texts
  42/140 student-texts
  56/140 student-texts
  70/140 student-texts
  84/140 student-texts
  98/140 student-texts
  112/140 student-texts
  126/140 student-texts
  140/140 student-texts
[done] answered=643 dropped=37 (5.44%) usage={'calls': 783, 'cache_hits': 0, 'input_tokens': 293205, 'output_tokens': 33159} -> runs_pilot/d1000
d1000 | dropped_stage1 8 | dropped_answers 0 | seconds 122.1 | {'calls': 783, 'cache_hits': 0, 'input_tokens': 293205, 'output_tokens': 33159}
d400 | dropped_stage1 6 | dropp

## Stage 2: second seed for noise (random, baseline, SPR; about 50 min)

In [8]:
!python scripts/02_run_simulation.py --mode random --students 300 --seed 1
!python scripts/02_run_simulation.py --mode baseline --students 300 --seed 1 {COMMON}
!python scripts/02_run_simulation.py --mode spr --students 300 --seed 1 {COMMON}
show('runs/*-s1/meta.json')

[run] random-c4-none-n300-s1: 300 students x ~7 texts
  210/2100 student-texts
  420/2100 student-texts
  630/2100 student-texts
  840/2100 student-texts
  1050/2100 student-texts
  1260/2100 student-texts
  1470/2100 student-texts
  1680/2100 student-texts
  1890/2100 student-texts
  2100/2100 student-texts
[done] answered=10205 dropped=0 (0.00%) usage=None -> runs/random-c4-none-n300-s1
[run] baseline-c4-local-n300-s1: 300 students x ~7 texts
  210/2100 student-texts
  420/2100 student-texts
  630/2100 student-texts
  840/2100 student-texts
  1050/2100 student-texts
  1260/2100 student-texts
  1470/2100 student-texts
  1680/2100 student-texts
  1890/2100 student-texts
  2100/2100 student-texts
[done] answered=10205 dropped=0 (0.00%) usage={'calls': 10205, 'cache_hits': 0, 'input_tokens': 9231073, 'output_tokens': 70400} -> runs/baseline-c4-local-n300-s1
[run] spr-c4-local-n300-s1: 300 students x ~7 texts
  210/2100 student-texts
  420/2100 student-texts
  630/2100 student-texts
  840

In [9]:
export()

exported -> /kaggle/working/outputs.zip (download it now)


## Stage 3: capacity sweep, SPR with C = 2, 6, 8, 12 (about 2 h; C=4 comes from session 1)

In [10]:
for c in [2,6,8,12]:
    !python scripts/02_run_simulation.py --mode spr --capacity {c} --students 300 {COMMON}
show('runs/spr-c*-local-n300-s0/meta.json')

[run] spr-c2-local-n300-s0: 300 students x ~7 texts
  210/2100 student-texts
  420/2100 student-texts
  630/2100 student-texts
  840/2100 student-texts
  1050/2100 student-texts
  1260/2100 student-texts
  1470/2100 student-texts
  1680/2100 student-texts
  1890/2100 student-texts
  2100/2100 student-texts
[done] answered=9592 dropped=804 (7.73%) usage={'calls': 11730, 'cache_hits': 0, 'input_tokens': 4244950, 'output_tokens': 183095} -> runs/spr-c2-local-n300-s0
[run] spr-c6-local-n300-s0: 300 students x ~7 texts
  210/2100 student-texts
  420/2100 student-texts
  630/2100 student-texts
  840/2100 student-texts
  1050/2100 student-texts
  1260/2100 student-texts
  1470/2100 student-texts
  1680/2100 student-texts
  1890/2100 student-texts
  2100/2100 student-texts
[done] answered=10186 dropped=210 (2.02%) usage={'calls': 12295, 'cache_hits': 0, 'input_tokens': 5022089, 'output_tokens': 317088} -> runs/spr-c6-local-n300-s0
[run] spr-c8-local-n300-s0: 300 students x ~7 texts
  210/2100 

In [11]:
export()

exported -> /kaggle/working/outputs.zip (download it now)


## Stage 4: extension, per-student capacity drawn uniformly from 2..6 (about 30 min)

In [12]:
!python scripts/02_run_simulation.py --mode spr --capacity-range 2 6 --students 300 {COMMON} --name spr-crange2to6-local-n300-s0
show('runs/spr-crange*/meta.json')

[run] spr-crange2to6-local-n300-s0: 300 students x ~7 texts
  210/2100 student-texts
  420/2100 student-texts
  630/2100 student-texts
  840/2100 student-texts
  1050/2100 student-texts
  1260/2100 student-texts
  1470/2100 student-texts
  1680/2100 student-texts
  1890/2100 student-texts
  2100/2100 student-texts
[done] answered=10056 dropped=294 (2.84%) usage={'calls': 12172, 'cache_hits': 372, 'input_tokens': 4554451, 'output_tokens': 246181} -> runs/spr-crange2to6-local-n300-s0
spr-crange2to6-local-n300-s0 | dropped_stage1 54 | dropped_answers 16 | seconds 1692.9 | {'calls': 12172, 'cache_hits': 372, 'input_tokens': 4554451, 'output_tokens': 246181}


In [13]:
export()

exported -> /kaggle/working/outputs.zip (download it now)


## Stage 5 (optional, only if quota/time remain): second seed for TS and dropout (about 1.5 h)

In [14]:
for mode in ['ts','spr_dropout']:
    !python scripts/02_run_simulation.py --mode {mode} --students 300 --seed 1 {COMMON}
show('runs/ts-*-s1/meta.json'); show('runs/spr_dropout-*-s1/meta.json')

[run] ts-c2-local-n300-s1: 300 students x ~7 texts
  210/2100 student-texts
  420/2100 student-texts
  630/2100 student-texts
  840/2100 student-texts
  1050/2100 student-texts
  1260/2100 student-texts
  1470/2100 student-texts
  1680/2100 student-texts
  1890/2100 student-texts
  2100/2100 student-texts
[done] answered=10060 dropped=145 (1.42%) usage={'calls': 20286, 'cache_hits': 0, 'input_tokens': 12791686, 'output_tokens': 599039} -> runs/ts-c2-local-n300-s1
[run] spr_dropout-c4-local-n300-s1: 300 students x ~7 texts
  210/2100 student-texts
  420/2100 student-texts
  630/2100 student-texts
  840/2100 student-texts
  1050/2100 student-texts
  1260/2100 student-texts
  1470/2100 student-texts
  1680/2100 student-texts
  1890/2100 student-texts
  2100/2100 student-texts
[done] answered=9878 dropped=327 (3.20%) usage={'calls': 12026, 'cache_hits': 0, 'input_tokens': 4511904, 'output_tokens': 492838} -> runs/spr_dropout-c4-local-n300-s1
ts-c2-local-n300-s1 | dropped_stage1 124 | dropp

In [15]:
export()

exported -> /kaggle/working/outputs.zip (download it now)


## Finish: quick table, final export, clean up (licensed texts must not stay in saved output)

In [16]:
!python scripts/03_evaluate.py --no-authors

 source                         name  n_students  mean_acc  score_sd   gap   jsd   ece  pearson  spearman  n_responses  dropped_pct
authors  Humans (test, ground truth)        1197     0.685     0.173   NaN   NaN   NaN      NaN       NaN          NaN          NaN
authors            Humans (held-out)        1161     0.661     0.181 0.026 0.006 0.027    0.895     0.845      35393.0          NaN
   mine    baseline-c4-local-n300-s0         300     0.897     0.056 0.210 0.451 0.214    0.267     0.219      10396.0        0.000
   mine    baseline-c4-local-n300-s1         300     0.895     0.054 0.208 0.446 0.213    0.262     0.213      10205.0        0.000
   mine       random-c4-none-n300-s0         300     0.366     0.089 0.321 0.627 0.316    0.354     0.319      10396.0        0.000
   mine       random-c4-none-n300-s1         300     0.376     0.086 0.311 0.619 0.303    0.315     0.253      10205.0        0.000
   mine        spr-c12-local-n300-s0         300     0.706     0.085 0.019 0

In [17]:
export()
proc.terminate()
for p in ['/tmp/cache.jsonl','/tmp/authors_data','runs_pilot']:
    shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else (os.path.exists(p) and os.remove(p))
print('done; cache and authors data removed. Stop the session now.')

exported -> /kaggle/working/outputs.zip (download it now)
done; cache and authors data removed. Stop the session now.
